# 02. EDA - Cookie Cats A/B Testing Dataset

모바일 게임 리텐션 데이터 탐색 + A/B 테스트 통계적 유의성 검증

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 로드 & 기본 정보

In [ ]:
from src.data.loader import load_cookie_cats

df = load_cookie_cats()

print(f"데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}컬럼")
print(f"\n컬럼 정보:")
print(f"  version: A/B 그룹 (gate_30 vs gate_40)")
print(f"  sum_gamerounds: 설치 후 14일간 총 플레이 라운드 수")
print(f"  retention_1: 설치 다음날 복귀 여부")
print(f"  retention_7: 설치 7일 후 복귀 여부")
print(f"\n그룹별 유저 수:")
print(df['version'].value_counts())
print(f"\n리텐션 현황:")
print(f"  1일 리텐션: {df['retention_1'].mean():.2%}")
print(f"  7일 리텐션: {df['retention_7'].mean():.2%}")
df.head()

## 2. 게임 라운드 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 전체 분포 (극단값 많으므로 상위 1% 제외)
cap = df['sum_gamerounds'].quantile(0.99)
df_capped = df[df['sum_gamerounds'] <= cap]

axes[0].hist(df_capped['sum_gamerounds'], bins=50, color='#3498db', edgecolor='white')
axes[0].set_title(f'게임 라운드 분포 (상위 1% 제외, cap={cap:.0f})')
axes[0].set_xlabel('sum_gamerounds')
axes[0].set_ylabel('유저 수')
axes[0].axvline(df['sum_gamerounds'].median(), color='red', linestyle='--', label=f'중앙값: {df["sum_gamerounds"].median():.0f}')
axes[0].legend()

# A/B 그룹별 비교
for version, color in [('gate_30', '#e74c3c'), ('gate_40', '#2ecc71')]:
    subset = df_capped[df_capped['version'] == version]
    axes[1].hist(subset['sum_gamerounds'], bins=50, alpha=0.5, color=color, label=version, density=True)
axes[1].set_title('A/B 그룹별 게임 라운드 분포')
axes[1].set_xlabel('sum_gamerounds')
axes[1].set_ylabel('밀도')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n기본 통계:")
print(df['sum_gamerounds'].describe())

## 3. A/B 그룹별 리텐션 비교

In [ ]:
retention_by_group = df.groupby('version')[['retention_1', 'retention_7']].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, ret_col in enumerate(['retention_1', 'retention_7']):
    ax = axes[i]
    day_label = '1일' if ret_col == 'retention_1' else '7일'
    vals = retention_by_group[ret_col]
    bars = ax.bar(vals.index, vals.values, color=['#e74c3c', '#2ecc71'])
    ax.set_title(f'{day_label} 리텐션: A/B 그룹 비교')
    ax.set_ylabel('리텐션율')
    ax.set_ylim(0, max(vals.values) * 1.3)
    
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.2%}', ha='center', fontsize=12, fontweight='bold')
    
    diff = vals['gate_30'] - vals['gate_40']
    ax.annotate(f'차이: {diff:+.2%}p', xy=(0.5, 0.85), xycoords='axes fraction',
                ha='center', fontsize=11, color='navy',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

plt.tight_layout()
plt.show()

print("\n그룹별 리텐션 요약:")
print(retention_by_group.to_string(float_format='{:.4%}'.format))

## 4. A/B 테스트 통계적 유의성 검증

In [ ]:
gate_30 = df[df['version'] == 'gate_30']
gate_40 = df[df['version'] == 'gate_40']

print("=" * 70)
print("A/B 테스트 통계적 유의성 검증")
print("=" * 70)
print(f"\n그룹 크기: gate_30={len(gate_30):,}, gate_40={len(gate_40):,}")

for ret_col, day_label in [('retention_1', '1일'), ('retention_7', '7일')]:
    print(f"\n--- {day_label} 리텐션 ---")
    
    # 비율
    p1 = gate_30[ret_col].mean()
    p2 = gate_40[ret_col].mean()
    n1 = len(gate_30)
    n2 = len(gate_40)
    
    print(f"  gate_30: {p1:.4%} ({gate_30[ret_col].sum():,}/{n1:,})")
    print(f"  gate_40: {p2:.4%} ({gate_40[ret_col].sum():,}/{n2:,})")
    print(f"  차이: {p1 - p2:+.4%}p")
    
    # Z-test for proportions
    p_pool = (gate_30[ret_col].sum() + gate_40[ret_col].sum()) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z_stat = (p1 - p2) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    print(f"\n  Two-proportion Z-test:")
    print(f"    Z-statistic: {z_stat:.4f}")
    print(f"    p-value: {p_value:.6f}")
    sig = "유의함 (p<0.05)" if p_value < 0.05 else "유의하지 않음 (p>=0.05)"
    print(f"    결론: {sig}")
    
    # 효과 크기 (Cohen's h)
    h = 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))
    print(f"    Cohen's h (효과 크기): {h:.4f}", end="")
    if abs(h) < 0.2:
        print(" (작음)")
    elif abs(h) < 0.5:
        print(" (보통)")
    else:
        print(" (큼)")

    # 95% 신뢰구간
    se_diff = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    ci_lower = (p1 - p2) - 1.96 * se_diff
    ci_upper = (p1 - p2) + 1.96 * se_diff
    print(f"    95% CI for difference: [{ci_lower:+.4%}, {ci_upper:+.4%}]")

## 5. 부트스트랩 시뮬레이션으로 검증

In [ ]:
# 부트스트랩으로 7일 리텐션 차이의 분포 추정
np.random.seed(42)
n_boot = 10000
boot_diffs = []

for _ in range(n_boot):
    boot_30 = gate_30['retention_7'].sample(n=len(gate_30), replace=True).mean()
    boot_40 = gate_40['retention_7'].sample(n=len(gate_40), replace=True).mean()
    boot_diffs.append(boot_30 - boot_40)

boot_diffs = np.array(boot_diffs)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(boot_diffs, bins=50, color='#3498db', edgecolor='white', alpha=0.7)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='차이=0 (효과 없음)')
ax.axvline(x=np.percentile(boot_diffs, 2.5), color='orange', linestyle='--', label='95% CI')
ax.axvline(x=np.percentile(boot_diffs, 97.5), color='orange', linestyle='--')
ax.set_title('부트스트랩: 7일 리텐션 차이 분포 (gate_30 - gate_40)')
ax.set_xlabel('리텐션율 차이')
ax.set_ylabel('빈도')
ax.legend()
plt.tight_layout()
plt.show()

ci_lower = np.percentile(boot_diffs, 2.5)
ci_upper = np.percentile(boot_diffs, 97.5)
print(f"부트스트랩 95% CI: [{ci_lower:+.4%}, {ci_upper:+.4%}]")
print(f"평균 차이: {boot_diffs.mean():+.4%}")
print(f"차이 > 0인 비율: {(boot_diffs > 0).mean():.1%}")

## 6. 게임 라운드 구간별 리텐션 분석

In [ ]:
# 게임 라운드 구간 생성
df['rounds_bin'] = pd.cut(df['sum_gamerounds'], 
                          bins=[0, 1, 10, 50, 100, 500, df['sum_gamerounds'].max()],
                          labels=['0-1', '2-10', '11-50', '51-100', '101-500', '500+'])

retention_by_rounds = df.groupby('rounds_bin')[['retention_1', 'retention_7']].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

retention_by_rounds['retention_1'].plot(kind='bar', ax=axes[0], color='#3498db')
axes[0].set_title('게임 라운드 구간별 1일 리텐션')
axes[0].set_ylabel('리텐션율')
axes[0].set_xlabel('게임 라운드 구간')
axes[0].tick_params(axis='x', rotation=30)
for i, val in enumerate(retention_by_rounds['retention_1']):
    axes[0].text(i, val + 0.01, f'{val:.1%}', ha='center', fontsize=9)

retention_by_rounds['retention_7'].plot(kind='bar', ax=axes[1], color='#e74c3c')
axes[1].set_title('게임 라운드 구간별 7일 리텐션')
axes[1].set_ylabel('리텐션율')
axes[1].set_xlabel('게임 라운드 구간')
axes[1].tick_params(axis='x', rotation=30)
for i, val in enumerate(retention_by_rounds['retention_7']):
    axes[1].text(i, val + 0.01, f'{val:.1%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# 유저 수 분포
print("\n구간별 유저 수:")
print(df['rounds_bin'].value_counts().sort_index())

## 7. 구간별 A/B 그룹 리텐션 비교

In [ ]:
# 구간별로 A/B 그룹 리텐션 차이가 달라지는지
cross_ret7 = df.groupby(['rounds_bin', 'version'])['retention_7'].mean().unstack()

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(cross_ret7))
width = 0.35
bars1 = ax.bar(x - width/2, cross_ret7['gate_30'], width, label='gate_30', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, cross_ret7['gate_40'], width, label='gate_40', color='#2ecc71', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(cross_ret7.index, rotation=30)
ax.set_title('게임 라운드 구간별 7일 리텐션: A/B 그룹 비교')
ax.set_ylabel('7일 리텐션율')
ax.set_xlabel('게임 라운드 구간')
ax.legend()
plt.tight_layout()
plt.show()

print("\n구간별 A/B 리텐션 차이 (gate_30 - gate_40):")
diff = cross_ret7['gate_30'] - cross_ret7['gate_40']
for bin_label, d in diff.items():
    print(f"  {bin_label:>10s}: {d:+.2%}p")

## 8. 핵심 인사이트 정리

### EDA 결론

**데이터 특성:**
- 90,189명 유저, A/B 그룹 거의 동일 크기
- 1일 리텐션 ~44.5%, 7일 리텐션 ~18.6%
- sum_gamerounds는 강한 right-skew (대부분 유저가 적은 라운드)

**A/B 테스트 결과:**
- (위 분석 실행 후 Z-test / 부트스트랩 결과 기반으로 작성)
- gate_30 vs gate_40의 리텐션 차이와 통계적 유의성

**본 프로젝트에서의 활용:**
- Cookie Cats 데이터는 "실제 리텐션 라벨이 있는 데이터"로 모델 교차 검증용
- A/B 테스트 통계 분석 역량을 보여주는 보조 자료
- sum_gamerounds 하나만으로는 피처가 부족 → 메인 모델링은 Gaming Behavior로